In [2]:
import json
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression, RidgeClassifier
from sklearn.metrics import f1_score, classification_report
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# ============================================================================
# 1. DATA LOADING
# ============================================================================
def load_data(train_path, test_path=None):
    """Load embeddings data"""
    print("Loading training data...")
    with open(train_path, 'r') as f:
        train_data = json.load(f)
    
    X_train = []
    y_train = []
    for record in train_data:
        features = record['image_embedding'] + record['text_embedding']
        X_train.append(features)
        y_train.append(record['label'])
    
    X_train = np.array(X_train)
    y_train = np.array(y_train)
    
    print(f"Training samples: {len(X_train)}")
    print(f"Features: {X_train.shape[1]}")
    print(f"Class 0: {sum(y_train==0)}, Class 1: {sum(y_train==1)}")
    
    X_test = None
    test_ids = None
    if test_path:
        print("Loading test data...")
        with open(test_path, 'r') as f:
            test_data = json.load(f)
        
        X_test = []
        test_ids = []
        for record in test_data:
            features = record['image_embedding'] + record['text_embedding']
            X_test.append(features)
            test_ids.append(record['id'])
        
        X_test = np.array(X_test)
        print(f"Test samples: {len(X_test)}")
    
    return X_train, y_train, X_test, test_ids

# ============================================================================
# 2. TRAIN MODELS (SIMPLE)
# ============================================================================
def train_models_simple(X_train, y_train, X_val, y_val):
    """Train 3 linear models with basic parameters"""
    
    print("\n" + "="*80)
    print("TRAINING LINEAR MODELS (SIMPLIFIED)")
    print("="*80)
    
    # Scale features
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)
    
    # Model 1: Logistic Regression
    print("\n1. Training Logistic Regression...")
    lr = LogisticRegression(
        C=1.0,
        max_iter=1000,
        class_weight='balanced',
        random_state=42
    )
    lr.fit(X_train_scaled, y_train)
    lr_pred = lr.predict(X_val_scaled)
    lr_f1 = f1_score(y_val, lr_pred, average='macro')
    print(f"   Validation F1: {lr_f1:.4f}")
    
    # Model 2: Ridge Classifier
    print("\n2. Training Ridge Classifier...")
    ridge = RidgeClassifier(
        alpha=1.0,
        class_weight='balanced',
        random_state=42
    )
    ridge.fit(X_train_scaled, y_train)
    ridge_pred = ridge.predict(X_val_scaled)
    ridge_f1 = f1_score(y_val, ridge_pred, average='macro')
    print(f"   Validation F1: {ridge_f1:.4f}")
    
    # Model 3: ElasticNet (via Logistic Regression)
    print("\n3. Training ElasticNet...")
    elastic = LogisticRegression(
        penalty='elasticnet',
        C=1.0,
        l1_ratio=0.5,
        solver='saga',
        max_iter=2000,
        class_weight='balanced',
        random_state=42
    )
    elastic.fit(X_train_scaled, y_train)
    elastic_pred = elastic.predict(X_val_scaled)
    elastic_f1 = f1_score(y_val, elastic_pred, average='macro')
    print(f"   Validation F1: {elastic_f1:.4f}")
    
    # Ensemble with simple averaging
    print("\n" + "="*80)
    print("CREATING ENSEMBLE")
    print("="*80)
    
    lr_probs = lr.predict_proba(X_val_scaled)
    
    # Convert Ridge scores to probabilities
    ridge_scores = ridge.decision_function(X_val_scaled)
    ridge_probs = np.column_stack([1 - ridge_scores, ridge_scores])
    ridge_probs = np.exp(ridge_probs) / np.exp(ridge_probs).sum(axis=1, keepdims=True)
    
    elastic_probs = elastic.predict_proba(X_val_scaled)
    
    # Simple average ensemble
    ensemble_probs = (lr_probs + ridge_probs + elastic_probs) / 3
    ensemble_pred = np.argmax(ensemble_probs, axis=1)
    ensemble_f1 = f1_score(y_val, ensemble_pred, average='macro')
    
    print(f"\nLogistic Regression F1: {lr_f1:.4f}")
    print(f"Ridge Classifier F1:    {ridge_f1:.4f}")
    print(f"ElasticNet F1:          {elastic_f1:.4f}")
    print(f"Ensemble F1:            {ensemble_f1:.4f}")
    
    print("\nEnsemble Classification Report:")
    print(classification_report(y_val, ensemble_pred, 
                              target_names=['Not Important', 'Important']))
    
    return lr, ridge, elastic, scaler, ensemble_f1

# ============================================================================
# 3. PREDICT ON TEST SET
# ============================================================================
def predict_test(lr, ridge, elastic, scaler, X_test, test_ids):
    """Make predictions on test set"""
    
    print("\n" + "="*80)
    print("PREDICTING ON TEST SET")
    print("="*80)
    
    X_test_scaled = scaler.transform(X_test)
    
    # Get predictions from all models
    lr_probs = lr.predict_proba(X_test_scaled)
    
    ridge_scores = ridge.decision_function(X_test_scaled)
    ridge_probs = np.column_stack([1 - ridge_scores, ridge_scores])
    ridge_probs = np.exp(ridge_probs) / np.exp(ridge_probs).sum(axis=1, keepdims=True)
    
    elastic_probs = elastic.predict_proba(X_test_scaled)
    
    # Simple average ensemble
    ensemble_probs = (lr_probs + ridge_probs + elastic_probs) / 3
    ensemble_pred = np.argmax(ensemble_probs, axis=1)
    
    print(f"Predictions: Class 0={sum(ensemble_pred==0)}, Class 1={sum(ensemble_pred==1)}")
    print(f"Positive class: {sum(ensemble_pred==1)/len(ensemble_pred)*100:.1f}%")
    
    submission = pd.DataFrame({
        'row_id': test_ids,
        'target': ensemble_pred
    })
    
    return submission

# ============================================================================
# 4. MAIN PIPELINE
# ============================================================================
def main():
    """Main pipeline"""
    
    print("="*80)
    print("SIMPLE LINEAR ENSEMBLE PIPELINE")
    print("="*80)
    
    # Load data
    X_train, y_train, X_test, test_ids = load_data(
        'D:/DSS-comp/train_part2.json',
        'D:/DSS-comp/test.json'
    )
    
    # Split for validation
    X_train_split, X_val_split, y_train_split, y_val_split = train_test_split(
        X_train, y_train,
        test_size=0.2,
        random_state=42,
        stratify=y_train
    )
    
    # Train models
    lr, ridge, elastic, scaler, val_f1 = train_models_simple(
        X_train_split, y_train_split,
        X_val_split, y_val_split
    )
    
    # Retrain on full data
    print("\n" + "="*80)
    print("RETRAINING ON FULL DATA")
    print("="*80)
    
    scaler_final = StandardScaler()
    X_train_final = scaler_final.fit_transform(X_train)
    
    lr_final = LogisticRegression(
        C=1.0, max_iter=1000, class_weight='balanced', random_state=42
    )
    lr_final.fit(X_train_final, y_train)
    print("✓ Logistic Regression trained")
    
    ridge_final = RidgeClassifier(
        alpha=1.0, class_weight='balanced', random_state=42
    )
    ridge_final.fit(X_train_final, y_train)
    print("✓ Ridge Classifier trained")
    
    elastic_final = LogisticRegression(
        penalty='elasticnet', C=1.0, l1_ratio=0.5,
        solver='saga', max_iter=2000, class_weight='balanced', random_state=42
    )
    elastic_final.fit(X_train_final, y_train)
    print("✓ ElasticNet trained")
    
    # Predict on test set
    submission = predict_test(
        lr_final, ridge_final, elastic_final,
        scaler_final, X_test, test_ids
    )
    
    # Save submission
    filename = f'logistic_v5.csv'
    submission.to_csv(filename, index=False)
    
    print(f"\n✓ Submission saved: {filename}")
    print(f"\nExpected F1 Score: {val_f1:.4f}")
    print("\n" + "="*80)
    print("DONE! (Much faster than the complex version)")
    print("="*80)

if __name__ == "__main__":
    main()

SIMPLE LINEAR ENSEMBLE PIPELINE
Loading training data...
Training samples: 1531
Features: 1024
Class 0: 1337, Class 1: 194
Loading test data...
Test samples: 500

TRAINING LINEAR MODELS (SIMPLIFIED)

1. Training Logistic Regression...
   Validation F1: 0.5860

2. Training Ridge Classifier...
   Validation F1: 0.5086

3. Training ElasticNet...
   Validation F1: 0.6362

CREATING ENSEMBLE

Logistic Regression F1: 0.5860
Ridge Classifier F1:    0.5086
ElasticNet F1:          0.6362
Ensemble F1:            0.6035

Ensemble Classification Report:
               precision    recall  f1-score   support

Not Important       0.90      0.91      0.91       268
    Important       0.32      0.28      0.30        39

     accuracy                           0.83       307
    macro avg       0.61      0.60      0.60       307
 weighted avg       0.82      0.83      0.83       307


RETRAINING ON FULL DATA
✓ Logistic Regression trained
✓ Ridge Classifier trained
✓ ElasticNet trained

PREDICTING ON TE